# 02 — Analisis Fitur & Lag Time-Series (Feature Analysis)
**NapasLoka: Pembentukan Lag Waktu, Skenario Fitur, dan Pemisahan Data Kronologis**

Notebook ini mendemonstrasikan tahapan rekayasa fitur (*Feature Engineering*) untuk peramalan deret waktu kualitas udara 1 jam ke depan ($t+1$):
1. **Analisis Autokorelasi (ACF)** untuk justifikasi lag waktu $t-1, t-2, t-3$.
2. **Pembentukan Fitur Skenario 1 vs Skenario 2** menggunakan `src.features`.
3. **Penanganan Baris Awal (Drop NaN)** agar data bebas kebocoran (*leakage-safe*).
4. **Pemisahan Kronologis 80% Train : 20% Test** dengan verifikasi tanggal transisi.


---
### 1. Setup Lingkungan & Import Modul


In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('napasloka'):
        !git clone https://github.com/Nikanzz/napasloka.git
    %cd napasloka
    !pip install -q xgboost lightgbm shap pyyaml joblib matplotlib scikit-learn statsmodels
    ROOT_DIR = Path("/content/napasloka")
else:
    ROOT_DIR = Path.cwd()
    if not (ROOT_DIR / "data").exists() and (ROOT_DIR.parent / "data").exists():
        ROOT_DIR = ROOT_DIR.parent

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.features.lag_features import create_lag_features
from src.features.scenarios import resolve_scenario
from src.data.preprocessing import chronological_train_test_split
from src.utils.config import load_yaml_config

print("✓ Modul feature engineering berhasil dimuat.")


---
### 2. Pemilihan Stasiun & Pemuatan Data


In [ ]:
STATION_ID = "pantai-makmur"
POLLUTANT = "co2"

df_raw = pd.read_csv(ROOT_DIR / "data" / "processed" / f"jabodetabek_{POLLUTANT}_dataset.csv")
df_raw["datetime"] = pd.to_datetime(df_raw["datetime"], utc=True).dt.tz_convert("Asia/Jakarta")

locations_cfg = load_yaml_config("locations.yaml").get("locations", [])
loc_info = next((l for l in locations_cfg if l["id"] == STATION_ID), None)
coord_id = f"{loc_info['latitude']}_{loc_info['longitude']}" if loc_info else STATION_ID

df_station = df_raw[(df_raw["location"] == coord_id) | (df_raw["location"] == STATION_ID)].sort_values("datetime").reset_index(drop=True)
print(f"Stasiun Terpilih : {loc_info['name'] if loc_info else STATION_ID} ({len(df_station)} baris)")
print(f"Rentang Waktu    : {df_station['datetime'].min()} s.d. {df_station['datetime'].max()}")


---
### 3. Analisis Autokorelasi Fitur Lag (ACF)
Grafik ACF menunjukkan signifikansi korelasi antara observasi terkini ($t$) dengan observasi jam-jam sebelumnya ($t-1, t-2, t-3$).


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plt.figure(figsize=(10, 4))
plot_acf(df_station["carbon_dioxide"].dropna(), lags=12, alpha=0.05)
plt.title("Autocorrelation Function (ACF) CO₂ - Hingga 12 Jam ke Belakang", fontsize=12)
plt.xlabel("Lag (Jam)")
plt.ylabel("Autokorelasi")
plt.grid(True, linestyle=":", alpha=0.6)
plt.show()


---
### 4. Pembentukan Fitur Lag: Skenario 1 vs Skenario 2


In [ ]:
# Skenario 1: Hanya lag polutan (t-1, t-2, t-3)
df_s1 = create_lag_features(df_station, pollutant=POLLUTANT, scenario="scenario_1", drop_incomplete_rows=True)
# Skenario 2: Lag polutan + Lag meteorologi (t-1, t-2, t-3)
df_s2 = create_lag_features(df_station, pollutant=POLLUTANT, scenario="scenario_2", drop_incomplete_rows=True)

s1_def = resolve_scenario(POLLUTANT, "scenario_1")
s2_def = resolve_scenario(POLLUTANT, "scenario_2")

print(f"Skenario 1 Fitur ({len(s1_def.feature_names)} kolom): {list(s1_def.feature_names)}")
print(f"Skenario 2 Fitur ({len(s2_def.feature_names)} kolom): {list(s2_def.feature_names)}")
print(f"Baris setelah eliminasi NaN awal: {len(df_s2)} baris (Observasi awal dimulai: {df_s2['datetime'].min()})")


---
### 5. Pembagian Kronologis 80% Train : 20% Test


In [ ]:
train_df, test_df = chronological_train_test_split(df_s2, test_fraction=0.2)

print("=== RINCIAN PEMBAGIAN DATA DERET WAKTU (80:20) ===")
print(f"Data Latih (80%): {len(train_df):,} baris | {train_df['datetime'].min()} s.d. {train_df['datetime'].max()}")
print(f"Data Uji   (20%): {len(test_df):,} baris | {test_df['datetime'].min()} s.d. {test_df['datetime'].max()}")

# Visualisasi Timeline Pembagian Data
plt.figure(figsize=(12, 2.5))
plt.plot(train_df["datetime"], np.ones(len(train_df)), color="#0284c7", linewidth=6, label=f"Train Set 80% ({len(train_df):,} sampel)")
plt.plot(test_df["datetime"], np.ones(len(test_df)), color="#ea580c", linewidth=6, label=f"Test Set 20% ({len(test_df):,} sampel)")
plt.title("Timeline Pembagian Kronologis Data Latih dan Data Uji", fontsize=12)
plt.yticks([])
plt.xlabel("Waktu Observasi (WIB)")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()
